## Modelo XGBoost

XGBoost (*Extreme Gradient Boosting*) es un algoritmo de machine learning supervisado basado en árboles de decisión. Construye un modelo predictivo a través de un proceso iterativo impulsado por la optimización del descenso del gradiente para minimizar la función de pérdida.

En una primera instancia, se entrenó un modelo XGBoost base incluyendo la totalidad de las variables de la temporada 1. Sin embargo, al evaluar su rendimiento sobre el conjunto de datos de la temporada 2, generó predicciones muy deficientes (obteniendo un LogLoss de aproximadamente 2.71).

Al analizar la importancia de las variables, detectamos que el modelo dependía casi exclusivamente de `sz_top` y `sz_bot` (y de las variables que se calculan a partir de ellas) para clasificar los swings.

Para lograr mejores predicciones, en esta notebook vamos a desarrollar y comparar dos modelos:

1. Un modelo excluyendo por completo las variables de la zona de strikes.
2. Un modelo ligeramente superador, implementando *feature engineering* para corregir espacialmente dichas variables.

Finalmente, para garantizar que los modelos alcancen su máximo rendimiento predictivo, implementaremos `FLAML` (Fast and Lightweight AutoML). Esta herramienta nos permitirá realizar una optimización bayesiana automatizada de los hiperparámetros, maximizando la eficiencia computacional y minimizando nuestra métrica de error.

### Carga de librerías

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from flaml import AutoML

### Carga de datos de la temporada 1

In [2]:
ROOT = Path("..")
ruta_t1 = ROOT / "data" / "processed" / "temporada1_con_nulos.parquet"
temporada1 = pl.read_parquet(ruta_t1).to_pandas()

## Primer modelo - Excluímos variables del sensor

### Definición de variables predictoras y respuesta

In [3]:
columnas_excluir = ["pitch_id", "batter", "pitcher", "description", "swing"]
columnas_contaminadas = [
    "sz_top",
    "sz_bot",
    "sz_mid",
    "strike_zone_size",
    "relative_height",
    "pitch_location",
    "is_strike_zone",
    "is_shadow_zone",
    "distance_to_corner",
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(
    columns=[col for col in todas_excluidas if col in temporada1.columns]
)
respuesta = temporada1["swing"]

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=["object", "string"]).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype("category")

print(
    f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras."
)

El dataset tiene 709852 filas y 15 variables predictoras.


### Configuración del modelo XGBoost

La ventaja de usar `FLAML` es que no solo realiza una búsqueda de los hiperparámetros mucho más inteligente que simplemente probar algunas combinaciones al azar, sino que también hace el *K-fold* y las particiones automáticamente.

In [4]:
# Inicializamos el motor de búsqueda
automl = AutoML()

# Configuramos la búsqueda del mejor modelo
opciones_flaml = {
    "time_budget": 600,  # Tiempo en segundos que le damos a FLAML para buscar
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["xgboost"],  # Optimiza un XGBoost
    "seed": 714,
    "verbose": 0,  # Que no imprima nada en la consola (aunque activándolo tampoco mostraba nada. Quizás es porque estamos trabajando en una nb)
}

automl.fit(X_train=explicativas, y_train=respuesta, **opciones_flaml)

# Resultados finales
print(f"Mejores hiperparámetros encontrados: {automl.best_config}")
print(f"Mejor LogLoss estimado: {automl.best_loss:.4f}")

# Extraemos el modelo ganador entrenado con el conjunto de datos completo de la temporada 1
mejor_modelo = automl.model.estimator

Mejores hiperparámetros encontrados: {'n_estimators': 2875, 'max_leaves': 24, 'min_child_weight': np.float64(23.810024928232124), 'learning_rate': np.float64(0.060438936441426394), 'subsample': np.float64(0.7232492552737307), 'colsample_bylevel': np.float64(0.459155442796822), 'colsample_bytree': np.float64(0.9443445854098466), 'reg_alpha': np.float64(6.2544323348363635), 'reg_lambda': np.float64(0.9728550358150281)}
Mejor LogLoss estimado: 0.4467


In [5]:
importancias = mejor_modelo.feature_importances_

df_importancias = pd.DataFrame(
    {"variable": explicativas.columns, "importancia": importancias}
)

df_importancias = df_importancias.sort_values(by="importancia", ascending=False)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10).to_string(index=False))

 == TOP 10 VARIABLES MÁS IMPORTANTES ==
           variable  importancia
              pfx_x     0.300972
movement_complexity     0.122361
      complex_speed     0.105754
            plate_z     0.096864
            strikes     0.068200
         pitch_type     0.060827
           p_throws     0.048648
              pfx_z     0.044630
      release_speed     0.039641
              stand     0.033041


### Predicción para Kaggle

Primero, leemos los datos limpios de la temporada 2.

In [11]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio.parquet"

temporada2 = pl.read_parquet(ruta_t2)

Ahora, aplicamos el modelo entrenado con la temporada 1 sobre el modelo de la temporada 2 para realizar las predicciones de swing.

In [12]:
temporada2 = temporada2.to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2["pitch_id"]

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.drop(
    columns=[col for col in todas_excluidas if col in temporada2.columns]
)

# Hay un error de categoría 1-3. Lo convertimos en nulo.
X_test["count"] = X_test["count"].replace("1-3", np.nan)

# Pasamos las variables necesarias a categóricas, con las mismas categorías que el training set
columnas_texto = explicativas.select_dtypes(include=["category"]).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(
        X_test[col], categories=explicativas[col].cat.categories
    )

# Reordenamos las columnas para que coincidan con el conjunto de la temporada 1
X_test = X_test[mejor_modelo.feature_names_in_]

# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame(
    {"pitch_id": pitch_ids_t2, "swing_probability": probabilidades_swing}
)

Por último, guardamos el archivo con las predicciones

In [ ]:
ruta_prediccion = ROOT / "reports" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

## Segundo modelo - Variables de sensor modificadas

In [20]:
HALF_PLATE_CM = 21.59

# Leemos nuevamente los datos
temporada1 = pl.read_parquet(ruta_t1).to_pandas()

# Calculamos la zona histórica solo con los datos de entrenamiento
zona_promedio = temporada1.groupby("batter")[["sz_top", "sz_bot"]].mean().reset_index()
zona_promedio = zona_promedio.rename(
    columns={"sz_top": "sz_top_mean", "sz_bot": "sz_bot_mean"}
)

# Cruzamos los promedios al dataset
temporada1 = temporada1.merge(zona_promedio, on="batter", how="left")

In [21]:
# Calculamos la geometría estática
temporada1["strike_zone_size_curada"] = (
    temporada1["sz_top_mean"] - temporada1["sz_bot_mean"]
)
temporada1["relative_height_curada"] = (
    temporada1["plate_z"] - temporada1["sz_bot_mean"]
) / temporada1["strike_zone_size_curada"]
temporada1["relative_x"] = temporada1["plate_x"] / HALF_PLATE_CM

In [22]:
# Variables espaciales seguras
temporada1["is_strike_zone_curada"] = (
    (temporada1["plate_x"] >= -HALF_PLATE_CM)
    & (temporada1["plate_x"] <= HALF_PLATE_CM)
    & (temporada1["relative_height_curada"] >= 0)
    & (temporada1["relative_height_curada"] <= 1)
).astype(int)

In [23]:
condicion_zona_t1 = temporada1["is_strike_zone_curada"] == 1
distancia_adentro_t1 = np.sqrt(
    (1 - temporada1["relative_x"].abs()) ** 2
    + np.minimum(
        temporada1["relative_height_curada"], 1 - temporada1["relative_height_curada"]
    )
    ** 2
)
temporada1["distance_to_corner_curada"] = np.where(
    condicion_zona_t1, distancia_adentro_t1, 0.0
)

In [24]:
# Eliminamos las variables base y las espaciales contaminadas originales
columnas_excluir = ["pitch_id", "batter", "pitcher", "description", "swing"]
columnas_contaminadas = [
    "sz_top",
    "sz_bot",
    "sz_mid",
    "strike_zone_size",
    "relative_height",
    "pitch_location",
    "is_strike_zone",
    "is_shadow_zone",
    "distance_to_corner",
    "sz_top_mean",
    "sz_bot_mean",
    "relative_x",  # Sacamos las intermedias para que no hagan ruido
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(
    columns=[col for col in todas_excluidas if col in temporada1.columns]
)
respuesta = temporada1["swing"]

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=["object", "string"]).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype("category")

print(
    f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras."
)

El dataset tiene 709852 filas y 18 variables predictoras.


In [25]:
# Inicializamos el motor de búsqueda
automl = AutoML()

# Configuramos la búsqueda del mejor modelo
opciones_flaml = {
    "time_budget": 600,  # Tiempo en segundos que le damos a FLAML para buscar
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["xgboost"],  # Optimiza un XGBoost
    "seed": 714,
    "verbose": 0,  # Que no imprima nada en la consola (aunque activándolo tampoco mostraba nada. Quizás es porque estamos trabajando en una nb)
}

automl.fit(X_train=explicativas, y_train=respuesta, **opciones_flaml)

# Mostramos los resultados finales
print("-" * 40)
print(f"Mejores hiperparámetros encontrados: {automl.best_config}")
print(f"Mejor LogLoss estimado: {automl.best_loss:.4f}")
print("-" * 40)

# Extraemos el modelo ganador entrenado con el conjunto de datos completo de la temporada 1
mejor_modelo = automl.model.estimator

----------------------------------------
Mejores hiperparámetros encontrados: {'n_estimators': 201, 'max_leaves': 154, 'min_child_weight': np.float64(54.59192406728625), 'learning_rate': np.float64(0.2087723635666093), 'subsample': np.float64(0.6652179043495202), 'colsample_bylevel': 1.0, 'colsample_bytree': np.float64(0.6433712968021359), 'reg_alpha': np.float64(0.0468787894421549), 'reg_lambda': np.float64(0.3070126218463928)}
Mejor LogLoss estimado: 0.4378
----------------------------------------


### Predicción para Kaggle

In [26]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio.parquet"
temporada2 = pl.read_parquet(ruta_t2).to_pandas()

# Usamos el zona_promedio que calculamos con la temporada 1
temporada2 = temporada2.merge(zona_promedio, on="batter", how="left")

Al modelo lo entramos con el promedio de 'sz_top' y 'sz_bot' por bateador. El problema es que el dataset de la temporada 2 puede tener jugadores nuevos para los que no vamos a contar con estas medidas en la temporada 1. Vamos a ver cuántos son estos nuevos jugadores.

In [27]:
# Filtramos el dataframe quedándonos solo con las filas que no cruzaron (jugadores nuevos)
filas_nuevos = temporada2[temporada2["sz_top_mean"].isna()]

# Conteos de filas
nulos_t2 = len(filas_nuevos)
total_filas_t2 = len(temporada2)

# Conteos de individuos únicos
jugadores_nuevos = filas_nuevos["batter"].nunique()
total_jugadores_t2 = temporada2["batter"].nunique()

print(
    f"Pitcheos de bateadores nuevos: {nulos_t2} de {total_filas_t2} ({nulos_t2 / total_filas_t2 * 100:.2f}%)"
)
print(
    f"Cantidad de bateadores nuevos: {jugadores_nuevos} de {total_jugadores_t2} bateadores totales en la T2"
)

Pitcheos de bateadores nuevos: 81435 de 708310 (11.50%)
Cantidad de bateadores nuevos: 178 de 693 bateadores totales en la T2


A los bateadores nuevos los vamos a imputar con medidas estándar.

In [28]:
temporada2["sz_top_mean"] = temporada2["sz_top_mean"].fillna(3.5 * 30.48)
temporada2["sz_bot_mean"] = temporada2["sz_bot_mean"].fillna(1.5 * 30.48)

temporada2["strike_zone_size_curada"] = (
    temporada2["sz_top_mean"] - temporada2["sz_bot_mean"]
)
temporada2["relative_height_curada"] = (
    temporada2["plate_z"] - temporada2["sz_bot_mean"]
) / temporada2["strike_zone_size_curada"]
temporada2["relative_x"] = temporada2["plate_x"] / HALF_PLATE_CM

temporada2["is_strike_zone_curada"] = (
    (temporada2["plate_x"] >= -HALF_PLATE_CM)
    & (temporada2["plate_x"] <= HALF_PLATE_CM)
    & (temporada2["relative_height_curada"] >= 0)
    & (temporada2["relative_height_curada"] <= 1)
).astype(int)

condicion_zona_t2 = temporada2["is_strike_zone_curada"] == 1
distancia_adentro_t2 = np.sqrt(
    (1 - temporada2["relative_x"].abs()) ** 2
    + np.minimum(
        temporada2["relative_height_curada"], 1 - temporada2["relative_height_curada"]
    )
    ** 2
)
temporada2["distance_to_corner_curada"] = np.where(
    condicion_zona_t2, distancia_adentro_t2, 0.0
)

In [29]:
# Guardamos los IDs para el archivo final
pitch_ids_t2 = temporada2["pitch_id"]

# Filtramos X_test
X_test = temporada2.drop(
    columns=[col for col in todas_excluidas if col in temporada2.columns]
)

# Alineamos las variables categóricas con las de entrenamiento
columnas_texto = explicativas.select_dtypes(include=["category"]).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(
        X_test[col], categories=explicativas[col].cat.categories
    )

X_test = X_test[mejor_modelo.feature_names_in_]

# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos y guardamos el DataFrame final
predicciones = pl.DataFrame(
    {"pitch_id": pitch_ids_t2, "swing_probability": probabilidades_swing}
)

ruta_prediccion = ROOT / "reports" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

Como este modelo dio una predicción sobre el dataset de la temporada 2 ligeramente mejor al primer modelo XGBoost, vamos a deternos a analizar cuáles fueron las principales variables que utilizó para clasificar.

In [30]:
importancias = mejor_modelo.feature_importances_

df_importancias = pd.DataFrame(
    {"variable": explicativas.columns, "importancia": importancias}
)

df_importancias = df_importancias.sort_values(by="importancia", ascending=False)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10).to_string(index=False))

 == TOP 10 VARIABLES MÁS IMPORTANTES ==
                 variable  importancia
    is_strike_zone_curada     0.456884
                    pfx_x     0.121655
distance_to_corner_curada     0.082792
               pitch_type     0.059690
                  strikes     0.054323
   relative_height_curada     0.038878
                  plate_z     0.035312
                    pfx_z     0.021421
                 p_throws     0.019237
                    stand     0.017253


- `is_strike_zone_curada` (45.69%): la decisión de hacer swing o no depende en gran parte de si la pelota va a la zona válida o no.

- `pfx_x` (12.17%): un lanzamiento con mucho movimiento horizontal engaña al bateador y tiene influencia en la probabilidad de swing.

- `distance_to_corner_curada` (8.28%): el modelo detectó que las pelotas que llegan en las esquinas de la zona de strike generan una duda o un comportamiento distinto en el bateador respecto a las pelotas que van cómodas al centro.

En resumen, la decisión de batear o no depende de si la pelota va a la zona válida (*is_strike_zone_curada*), pero está también muy condicionada por cuánto se mueve la pelota de forma horizontal al llegar a la zona (*pfx_x*) y qué tan cerca del límite pasa (*distance_to_corner_curada*).